# Full evaluation of the heads

Complete test set, with no image limit and no border cropping. Loads both heads of each class (image and pixel) from a `train.py` run and applies the inference recipe stored in the pixel-head checkpoint.

This is a **descriptive** evaluation: the project only has train and test splits, and the test set was already used to select the epoch of the heads (and the levels, when the ablation ran). Thresholds and best F1 values are test oracles.

Select the kernel of the project environment. Edit the paths in the next cell and run the cells in order.

**Precision and cost:** `histogram` goes through every pixel but approximates AUROC/AP and thresholds with bins. `exact` uses sklearn over every pixel and may need many GB of RAM. Global AUPRO follows the CFLOW-AD implementation (`core.metrics`). NF scores are not calibrated probabilities.

In [ ]:
from pathlib import Path
import sys
from datetime import datetime

candidates = [Path.cwd(), Path.cwd() / 'anomaly-detection-pesquisa' / 'differnet', *Path.cwd().parents]
PROJECT_ROOT = next((path for path in candidates if (path / 'core' / 'pixel_head.py').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Open the notebook from the differnet project or set PROJECT_ROOT.')
sys.path.insert(0, str(PROJECT_ROOT))
import config as c

DATASET_ROOT = Path(c.dataset_path)
# train.py run with <class>/image_head.pt and <class>/pixel_head.pt.
# None = most recent run in config.output_dir.
MODEL_DIR = None
# Alternative: explicit checkpoints per class, e.g. {'vari-grip': ('se.pt', 'head.pt')}.
EXPLICIT_MODELS = {}
CLASSES = list(c.classes)
RECIPE_OVERRIDE = None  # None = checkpoint recipe; or 'best_per_class', 'general', 'baseline'.
COMPUTE_IMAGE_HEAD = True  # SEDifferNet Image AUROC (64 rotations per image).
PIXEL_BACKEND = 'histogram'  # 'exact' may exceed the available RAM.
N_BINS = 65536
SAMPLE_PER_IMAGE = 1024
BOOTSTRAP = 200
SEED = 42
EXAMPLES_PER_CLASS = 6
COMPUTE_AUPRO = True
KEEP_MAPS = False
OUTPUT_DIR = PROJECT_ROOT / 'evaluation' / datetime.now().strftime('notebook_%Y%m%d_%H%M%S')
assert PIXEL_BACKEND in ('histogram', 'exact')
assert N_BINS >= 32 and SAMPLE_PER_IMAGE > 0 and BOOTSTRAP >= 0
print('Project:', PROJECT_ROOT)
print('Dataset:', DATASET_ROOT)
print('Output:', OUTPUT_DIR)
print('Pixel backend:', PIXEL_BACKEND, '| complete test set, stride=1, margin=0')

In [ ]:
import gc
import hashlib
import json
import math
import time
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from IPython.display import display, Markdown
from tqdm.auto import tqdm
from core.data import TestDataset
from core.evaluation import image_head_scores, iter_maps
from core.inference import AnomalyDetector
from core.metrics import PixelMetrics, ScoreRange
from core.pixel_head import image_score_from_map
from core.recipes import resolve_recipe
from core.utils import resolve_models

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Python:', sys.executable)
print('PyTorch:', torch.__version__, '| device:', DEVICE)
if DEVICE == 'cuda':
    print(torch.cuda.get_device_name(0))
pd.set_option('display.max_columns', 40)


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def normalized_path(path):
    return str(path).replace('\\', '/').lower()


def safe_ratio(numerator, denominator):
    return float(numerator / denominator) if denominator else float('nan')


def confusion_metrics(tp, fp, fn, tn):
    precision = safe_ratio(tp, tp + fp)
    recall = safe_ratio(tp, tp + fn)
    specificity = safe_ratio(tn, tn + fp)
    f1 = safe_ratio(2 * tp, 2 * tp + fp + fn)
    normal_f1 = safe_ratio(2 * tn, 2 * tn + fp + fn)
    iou = safe_ratio(tp, tp + fp + fn)
    normal_iou = safe_ratio(tn, tn + fp + fn)
    total = tp + fp + fn + tn
    weighted_f1 = safe_ratio(((tp + fn) * f1 if tp + fn else 0.) +
                             ((tn + fp) * normal_f1 if tn + fp else 0.), total)
    return dict(tp=int(tp), fp=int(fp), fn=int(fn), tn=int(tn),
                accuracy=safe_ratio(tp + tn, total), precision=precision, recall=recall,
                specificity=specificity, balanced_accuracy=(recall + specificity) / 2,
                f1=f1, dice=f1, f1_normal=normal_f1,
                f1_macro_binary=float(np.nanmean([f1, normal_f1])), f1_weighted=weighted_f1,
                f1_micro_binary=safe_ratio(tp + tn, total),
                iou=iou, iou_normal=normal_iou, miou=float(np.nanmean([iou, normal_iou])),
                fpr=safe_ratio(fp, fp + tn))


def curve_from_counts(positive, negative):
    tp = np.r_[0., np.cumsum(positive[::-1], dtype=np.float64)]
    fp = np.r_[0., np.cumsum(negative[::-1], dtype=np.float64)]
    recall = tp / positive.sum() if positive.sum() else np.full(tp.shape, np.nan)
    fpr = fp / negative.sum() if negative.sum() else np.full(fp.shape, np.nan)
    precision = np.divide(tp, tp + fp, out=np.ones(tp.shape), where=(tp + fp) > 0)
    f1 = np.divide(2 * tp, positive.sum() + tp + fp,
                   out=np.zeros(tp.shape), where=(positive.sum() + tp + fp) > 0)
    return dict(fpr=fpr, recall=recall, precision=precision, f1=f1,
                auroc=float(np.trapz(recall, fpr)) if positive.sum() and negative.sum() else np.nan,
                ap=float(np.sum(np.diff(recall) * precision[1:])) if positive.sum() else np.nan)


def binary_image_metrics(labels, scores, threshold):
    labels, scores = np.asarray(labels), np.asarray(scores)
    predicted = scores >= threshold
    result = confusion_metrics(np.sum(predicted & (labels == 1)), np.sum(predicted & (labels == 0)),
                               np.sum(~predicted & (labels == 1)), np.sum(~predicted & (labels == 0)))
    both = len(np.unique(labels)) == 2
    result.update(auroc=roc_auc_score(labels, scores) if both else np.nan,
                  ap=average_precision_score(labels, scores) if both else np.nan)
    return result


def fit_image_threshold(labels, scores):
    precision, recall, thresholds = precision_recall_curve(labels, scores)
    if not len(thresholds) or len(np.unique(labels)) < 2:
        return float(np.median(scores))
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    return float(thresholds[np.nanargmax(f1)])

In [ ]:
def pixel_histograms(maps, masks):
    rng = np.random.default_rng(SEED)
    samples = [np.asarray(score).ravel()[rng.integers(0, score.size, SAMPLE_PER_IMAGE)] for score in maps]
    sample = np.concatenate(samples)
    edges = np.unique(np.quantile(sample, np.linspace(0, 1, N_BINS + 1)[1:-1]))
    slots = len(edges) + 1
    positive, negative = np.zeros(slots), np.zeros(slots)
    group_size = max(1, math.ceil(slots / 2048))
    starts = np.arange(0, slots, group_size)
    boot_positive = np.zeros((len(maps), len(starts)), dtype=np.uint32)
    boot_negative = np.zeros_like(boot_positive)
    per_image = []
    for index, (score, mask) in enumerate(tqdm(zip(maps, masks), total=len(maps), desc='Full histograms')):
        labels = np.asarray(mask).ravel().astype(bool)
        bins = np.searchsorted(edges, np.asarray(score).ravel(), side='right')
        pos = np.bincount(bins[labels], minlength=slots)
        neg = np.bincount(bins[~labels], minlength=slots)
        positive += pos
        negative += neg
        boot_positive[index] = np.add.reduceat(pos, starts)
        boot_negative[index] = np.add.reduceat(neg, starts)
        per_image.append(curve_from_counts(pos, neg)['auroc'])
    curve = curve_from_counts(positive, negative)
    thresholds = np.r_[np.inf, edges[::-1], -np.inf]
    oracle_index = int(np.nanargmax(curve['f1']))
    threshold = float(thresholds[oracle_index])
    result = dict(pixel_auroc=curve['auroc'], pixel_ap=curve['ap'],
                  pixel_prevalence=safe_ratio(positive.sum(), positive.sum() + negative.sum()),
                  pixel_best_f1_test_oracle=float(curve['f1'][oracle_index]),
                  pixel_threshold_test_oracle=float(thresholds[oracle_index]),
                  pixel_threshold=threshold, threshold_source='test_oracle',
                  pixel_ranking_backend=PIXEL_BACKEND, pixel_bins=slots,
                  pixel_threshold_backend='histogram')
    if PIXEL_BACKEND == 'exact':
        labels = np.asarray(masks).reshape(-1)
        scores = np.asarray(maps).reshape(-1)
        if labels.min() != labels.max():
            result['pixel_auroc'] = float(roc_auc_score(labels, scores))
            result['pixel_ap'] = float(average_precision_score(labels, scores))
    return result, curve, thresholds, boot_positive, boot_negative, per_image, sample


def image_bootstrap(labels, scores, positive, negative):
    if not BOOTSTRAP:
        return {}
    rng = np.random.default_rng(SEED)
    groups = [np.flatnonzero(labels == label) for label in (0, 1)]
    values = []
    for _ in tqdm(range(BOOTSTRAP), desc='Per-image bootstrap'):
        indices = np.concatenate([rng.choice(group, len(group), replace=True) for group in groups if len(group)])
        curve = curve_from_counts(positive[indices].sum(axis=0), negative[indices].sum(axis=0))
        values.append([curve['auroc'], curve['ap'],
                       roc_auc_score(labels[indices], scores[indices]) if len(np.unique(labels[indices])) == 2 else np.nan])
    limits = np.nanpercentile(values, [2.5, 97.5], axis=0)
    return {f'{name}_ci_{bound}': float(limits[side, index])
            for index, name in enumerate(('pixel_auroc_hist2048', 'pixel_ap_hist2048', 'image_auroc'))
            for side, bound in enumerate(('low', 'high'))}


def resolve_artifact(class_name):
    if class_name in EXPLICIT_MODELS:
        image_path, pixel_path = EXPLICIT_MODELS[class_name]
    else:
        image_path, pixel_path = resolve_models(class_name, MODEL_DIR, str(PROJECT_ROOT / c.output_dir))
    payload = torch.load(pixel_path, map_location='cpu', weights_only=False)
    override = None
    if RECIPE_OVERRIDE is not None:
        override = resolve_recipe(str(DATASET_ROOT), class_name, RECIPE_OVERRIDE)['inference']
    return dict(class_name=class_name, checkpoint=str(pixel_path), encoder=str(image_path),
                payload=payload, override=override)


ARTIFACTS = [resolve_artifact(name) for name in CLASSES]
display(pd.DataFrame([dict(class_name=item['class_name'], head=item['checkpoint'], encoder=item['encoder'],
                           epoch=item['payload'].get('epoch'),
                           recipe=item['override'] or item['payload'].get('inference_recipe'))
                      for item in ARTIFACTS]))

In [ ]:
def save_figure(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches='tight')
    display(fig)
    plt.close(fig)


def plot_class_results(name, table, maps, masks, metrics, curve, thresholds, sample, folder):
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    axes[0, 0].plot(curve['fpr'], curve['recall'])
    axes[0, 0].plot([0, 1], [0, 1], '--', color='gray')
    axes[0, 0].set(xlabel='FPR', ylabel='TPR', title=f'{name}: pixel ROC ({PIXEL_BACKEND})')
    axes[0, 1].plot(curve['recall'], curve['precision'])
    axes[0, 1].axhline(metrics['pixel_prevalence'], color='gray', linestyle='--', label='Prevalence')
    axes[0, 1].set(xlabel='Recall', ylabel='Precision', title='Pixel precision-recall'); axes[0, 1].legend()
    for label, group in table.groupby('image_label'):
        axes[0, 2].hist(group['image_score'], bins=30, alpha=.6, label='Anomalous' if label else 'Normal')
    axes[0, 2].axvline(metrics['image_threshold'], color='black', linestyle='--')
    axes[0, 2].set(title='Per-image score (not a probability)', xlabel='Score', ylabel='Images'); axes[0, 2].legend()
    finite = np.isfinite(thresholds)
    axes[1, 0].plot(thresholds[finite], curve['f1'][finite])
    axes[1, 0].axvline(metrics['pixel_threshold'], color='black', linestyle='--')
    axes[1, 0].set(title='Pixel F1 versus threshold (test)', xlabel='Threshold', ylabel='F1')
    matrix = np.array([[metrics['pixel_tn'], metrics['pixel_fp']], [metrics['pixel_fn'], metrics['pixel_tp']]])
    image = axes[1, 1].imshow(np.log1p(matrix), cmap='Blues')
    for row in range(2):
        for column in range(2):
            axes[1, 1].text(column, row, f'{matrix[row, column]:,}', ha='center', va='center')
    axes[1, 1].set(xticks=[0, 1], yticks=[0, 1], xlabel='Predicted', ylabel='Actual', title='Pixel confusion (log1p color)')
    fig.colorbar(image, ax=axes[1, 1])
    axes[1, 2].hist(table['nll_ms'], bins=30)
    axes[1, 2].set(title='Backbone + NF + TTA, no I/O/post-processing', xlabel='ms per image', ylabel='Images')
    save_figure(fig, folder / 'curves_distributions_confusion.png')
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    image_curve_precision, image_curve_recall, _ = precision_recall_curve(table['image_label'], table['image_score'])
    from sklearn.metrics import roc_curve
    image_fpr, image_tpr, _ = roc_curve(table['image_label'], table['image_score'])
    axes[0].plot(image_fpr, image_tpr); axes[0].set(title='Per-image ROC (map)', xlabel='FPR', ylabel='TPR')
    axes[1].plot(image_curve_recall, image_curve_precision); axes[1].set(title='Per-image PR (map)', xlabel='Recall', ylabel='Precision')
    save_figure(fig, folder / 'image_curves.png')
    fp_cases = table[table.image_label == 0].sort_values('fp', ascending=False).head(2)
    fn_cases = table[table.image_label == 1].sort_values('recall', na_position='last').head(2)
    boundary = table.assign(distance=(table.image_score - metrics['image_threshold']).abs()).sort_values('distance').head(EXAMPLES_PER_CLASS)
    indices = list(dict.fromkeys([*fp_cases.index, *fn_cases.index, *boundary.index]))[:EXAMPLES_PER_CLASS]
    low, high = np.quantile(sample, [.01, .99])
    high = max(high, low + 1e-8)
    if not indices:
        return
    fig, axes = plt.subplots(len(indices), 5, figsize=(17, 3.1 * len(indices)), squeeze=False)
    for row, index in enumerate(indices):
        with Image.open(table.loc[index, 'path']) as original:
            rgb = np.asarray(original.convert('RGB').resize((maps.shape[2], maps.shape[1])))
        axes[row, 0].imshow(rgb)
        axes[row, 0].set_title(Path(table.loc[index, 'path']).name, fontsize=8)
        axes[row, 1].imshow(masks[index], cmap='gray', vmin=0, vmax=1); axes[row, 1].set_title('Ground truth')
        heat = axes[row, 2].imshow(maps[index], cmap='inferno', vmin=low, vmax=high); axes[row, 2].set_title('Raw score (class q1-q99)')
        axes[row, 3].imshow(rgb)
        axes[row, 3].imshow(maps[index], cmap='inferno', vmin=low, vmax=high, alpha=.5); axes[row, 3].set_title('Overlay')
        axes[row, 4].imshow(maps[index] >= metrics['pixel_threshold'], cmap='gray', vmin=0, vmax=1)
        axes[row, 4].set_title(f"Prediction | IoU={table.loc[index, 'iou']:.3f}")
        for axis in axes[row]:
            axis.set_axis_off()
    fig.colorbar(heat, ax=axes[:, 2].tolist(), fraction=.02, label='NF score; not a probability')
    fig.savefig(folder / 'heatmap_examples.png', dpi=150, bbox_inches='tight')
    display(fig); plt.close(fig)
    np.savez_compressed(folder / 'example_maps.npz', indices=indices, scores=np.asarray(maps[indices]), masks=np.asarray(masks[indices]))
    ranking = table.assign(image_error=table.image_label != table.image_prediction)
    ranking.sort_values(['image_error', 'fp', 'fn'], ascending=False).to_csv(folder / 'error_ranking.csv', index=False)

In [ ]:
def evaluate_artifact(artifact):
    name = artifact['class_name']
    folder = OUTPUT_DIR / name
    folder.mkdir(parents=True, exist_ok=True)
    declared_class = (artifact['payload'].get('training_recipe') or {}).get('class_name')
    if declared_class is not None and declared_class != name:
        raise ValueError(f'Head belongs to class {declared_class}, not {name}')
    detector = AnomalyDetector(artifact['encoder'], artifact['checkpoint'], DEVICE, artifact['override'])
    recipe, head = detector.recipe, detector.pixel_head
    print(detector.describe())
    testset = TestDataset(str(DATASET_ROOT), name, n_views=1)
    labels = testset.labels.astype(int)
    paths = [record[0] for record in testset.samples]
    size = c.img_size[0]
    dtype = np.float64 if recipe['score_mode'] == 'cflow_original' else np.float32
    files = [folder / 'native_nll.npy', folder / 'maps.npy', folder / 'masks.npy']
    native = np.lib.format.open_memmap(files[0], mode='w+', dtype=np.float32,
                                      shape=(len(testset), len(head.trained_levels),
                                             c.pixel_out_size, c.pixel_out_size))
    maps = np.lib.format.open_memmap(files[1], mode='w+', dtype=dtype, shape=(len(testset), size, size))
    masks = np.lib.format.open_memmap(files[2], mode='w+', dtype=np.uint8, shape=maps.shape)
    times = []
    started = time.perf_counter()
    try:
        with torch.inference_mode():
            for index in tqdm(range(len(testset)), desc=f'{name}: complete test set'):
                views, _ = testset[index]
                images = views[:1].to(DEVICE)
                if DEVICE == 'cuda':
                    torch.cuda.synchronize()
                begin = time.perf_counter()
                nll = head.nll(images)
                if DEVICE == 'cuda':
                    torch.cuda.synchronize()
                times.append((time.perf_counter() - begin) * 1000)
                if not torch.isfinite(nll).all():
                    raise ValueError(f'Non-finite NLL: {paths[index]}')
                native[index] = nll[0].numpy()
                masks[index] = testset.load_mask(index).astype(np.uint8)
        native.flush()
        for index, score in enumerate(tqdm(iter_maps(native, recipe, head.trained_levels, head.level_stats),
                                           total=len(testset), desc=f'{name}: aggregation')):
            maps[index] = score
        maps.flush(); masks.flush()
        if not all(np.isfinite(score).all() for score in maps):
            raise ValueError('Non-finite final map')
        metrics, curve, thresholds, boot_pos, boot_neg, pixel_auroc_images, sample = pixel_histograms(maps, masks)
        image_scores = np.array([image_score_from_map(score, recipe['image_score'], recipe.get('topk_percent', 1.))
                                 for score in maps])
        image_threshold = fit_image_threshold(labels, image_scores)
        metrics.update(image_threshold=image_threshold, image_threshold_source='test_oracle')
        rows = []
        for index, (score, mask) in enumerate(zip(maps, masks)):
            predicted, actual = score >= metrics['pixel_threshold'], mask.astype(bool)
            counts = confusion_metrics(np.sum(predicted & actual), np.sum(predicted & ~actual),
                                       np.sum(~predicted & actual), np.sum(~predicted & ~actual))
            rows.append(dict(index=index, path=paths[index], defect=Path(paths[index]).parent.name,
                             image_label=int(labels[index]), image_score=float(image_scores[index]),
                             image_prediction=int(image_scores[index] >= image_threshold),
                             pixel_auroc=pixel_auroc_images[index], nll_ms=times[index], **counts))
        table = pd.DataFrame(rows).set_index('index')
        totals = table[['tp', 'fp', 'fn', 'tn']].sum()
        metrics.update({f'pixel_{key}': value for key, value in confusion_metrics(**totals.to_dict()).items()})
        metrics.update({f'image_{key}': value for key, value in binary_image_metrics(labels, image_scores, image_threshold).items()})
        metrics.update(image_bootstrap(labels, image_scores, boot_pos, boot_neg))
        if COMPUTE_AUPRO:
            score_range = ScoreRange()
            for score in maps:
                score_range.add(score)
            aupro = PixelMetrics(score_range.lo, score_range.hi, aupro=True)
            for score, mask in zip(maps, masks):
                aupro.add(score, mask.astype(bool))
            metrics['aupro_cflow_global'] = aupro.result()['aupro']
        else:
            metrics['aupro_cflow_global'] = np.nan
        if COMPUTE_IMAGE_HEAD:
            rotations = TestDataset(str(DATASET_ROOT), name, n_views=c.n_transforms_test)
            se_scores = image_head_scores(detector.image_model, rotations, DEVICE)
            table['image_head_score'] = se_scores
            metrics['image_head_auroc'] = roc_auc_score(labels, se_scores) if len(np.unique(labels)) == 2 else np.nan
        metrics.update(class_name=name, n_images=len(testset), n_anomaly=int(labels.sum()),
                       checkpoint_epoch=artifact['payload'].get('epoch'), nll_ms_mean=float(np.mean(times)),
                       nll_ms_p95=float(np.percentile(times, 95)), evaluation_seconds=time.perf_counter() - started,
                       recipe=json.dumps(recipe), bootstrap_method='stratified_images_hist2048_not_training_seeds')
        metrics['pixel_ap_lift_over_prevalence'] = safe_ratio(metrics['pixel_ap'], metrics['pixel_prevalence'])
        table.to_csv(folder / 'per_image.csv')
        pd.DataFrame(dict(fpr=curve['fpr'], recall=curve['recall'], precision=curve['precision'],
                          f1=curve['f1'], threshold=thresholds)).to_csv(folder / 'pixel_curves.csv', index=False)
        metadata = dict(dataset=str(DATASET_ROOT), head=artifact['checkpoint'], encoder=artifact['encoder'],
                        head_sha256=sha256_file(artifact['checkpoint']), encoder_sha256=sha256_file(artifact['encoder']),
                        recipe=recipe, level_stats=head.level_stats,
                        level_ablation=artifact['payload'].get('level_ablation'), metric_backend=PIXEL_BACKEND,
                        n_bins=N_BINS, bootstrap=BOOTSTRAP, stride=1, margin=0, cohort='complete_test',
                        warning='descriptive evaluation; test set also used for epoch/level selection')
        (folder / 'evaluation_config.json').write_text(json.dumps(metadata, indent=2, default=str), encoding='utf-8')
        plot_class_results(name, table, maps, masks, metrics, curve, thresholds, sample, folder)
        return metrics, table, curve
    finally:
        for array in (native, maps, masks):
            array.flush()
            array._mmap.close()
        if not KEEP_MAPS:
            for path in files:
                path.unlink(missing_ok=True)
        del detector
        gc.collect()
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

## Run the full evaluation

The next cell runs inference and every enabled metric on **every image and every pixel of the test set**, one class at a time. It can take a while. Cached maps are deleted at the end of each class if `KEEP_MAPS=False`; examples, tables and curves are kept.

The thresholds (pixel and image) are the best-F1 thresholds on the test set itself: diagnostic oracles, not production thresholds. The CIs resample images (bins reduced to 2048 for the pixel CIs); they do not measure variation across training runs, seeds or recipe choice, and there is no CI for AUPRO.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS, PER_IMAGE, CURVES = [], [], {}
for artifact in ARTIFACTS:
    print('\n' + '=' * 80, '\n', artifact['class_name'])
    metrics, per_image, curve = evaluate_artifact(artifact)
    RESULTS.append(metrics)
    PER_IMAGE.append(per_image.assign(class_name=artifact['class_name']))
    CURVES[artifact['class_name']] = curve
    pd.DataFrame(RESULTS).to_csv(OUTPUT_DIR / 'results_per_class.csv', index=False)
    pd.concat(PER_IMAGE).to_csv(OUTPUT_DIR / 'per_image_all.csv', index=False)
    display(pd.DataFrame([metrics]).drop(columns='recipe').T.rename(columns={0: 'value'}))
print('Evaluation finished:', OUTPUT_DIR)

In [ ]:
results = pd.DataFrame(RESULTS).set_index('class_name')
images = pd.concat(PER_IMAGE).reset_index()
metric_columns = ['pixel_auroc', 'pixel_ap', 'pixel_prevalence', 'pixel_ap_lift_over_prevalence',
                  'aupro_cflow_global', 'pixel_accuracy', 'pixel_precision', 'pixel_recall',
                  'pixel_f1', 'pixel_f1_macro_binary', 'pixel_f1_micro_binary', 'pixel_f1_weighted',
                  'pixel_iou', 'pixel_miou', 'pixel_dice', 'image_auroc', 'image_ap',
                  'image_accuracy', 'image_precision', 'image_recall', 'image_f1', 'nll_ms_mean']
if COMPUTE_IMAGE_HEAD:
    metric_columns.insert(metric_columns.index('image_auroc'), 'image_head_auroc')
summary = results[metric_columns].copy()
summary.loc['MACRO_CLASSES'] = summary.mean(axis=0, numeric_only=True)
display(summary.style.format('{:.4f}', na_rep='not computed').background_gradient(cmap='YlGnBu', axis=0))
summary.to_csv(OUTPUT_DIR / 'summary_macro.csv')
ci_columns = [column for column in results if '_ci_' in column]
display(results[['n_images', 'n_anomaly', 'threshold_source', 'pixel_ranking_backend', *ci_columns]])

pooled_counts = images[['tp', 'fp', 'fn', 'tn']].sum()
pooled = confusion_metrics(**pooled_counts.to_dict())
micro = pd.DataFrame([dict(scope='PIXELS_MICRO_ALL_CLASSES', **pooled)])
display(micro)
micro.to_csv(OUTPUT_DIR / 'micro_confusion_metrics.csv', index=False)
print('AUROC/AP are not pooled across classes: scores have different scales. MACRO_CLASSES is the per-class mean.')

subgroups = []
for (class_name, defect), group in images.groupby(['class_name', 'defect']):
    counts = group[['tp', 'fp', 'fn', 'tn']].sum()
    subgroups.append(dict(class_name=class_name, defect=defect, n_images=len(group),
                          n_anomaly=int(group.image_label.sum()), **confusion_metrics(**counts.to_dict())))
subgroup_table = pd.DataFrame(subgroups)
display(subgroup_table)
subgroup_table.to_csv(OUTPUT_DIR / 'metrics_by_defect.csv', index=False)

images['score_quartile'] = images.groupby('class_name')['image_score'].transform(
    lambda series: pd.qcut(series.rank(method='first'), min(4, len(series)), labels=False))
score_groups = images.groupby(['class_name', 'score_quartile']).agg(
    n_images=('image_label', 'size'), anomaly_rate=('image_label', 'mean'),
    mean_score=('image_score', 'mean'), mean_latency_ms=('nll_ms', 'mean'))
display(score_groups)
score_groups.to_csv(OUTPUT_DIR / 'metrics_by_score_quartile.csv')
print('Quartiles are score ranges, not calibrated probability/confidence ranges.')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
results[['pixel_auroc', 'pixel_ap', 'aupro_cflow_global']].plot.bar(ax=axes[0, 0])
axes[0, 0].set(title='Ranking and region metrics', ylabel='Score', ylim=(0, 1))
results[['pixel_precision', 'pixel_recall', 'pixel_f1', 'pixel_iou']].plot.bar(ax=axes[0, 1])
axes[0, 1].set(title='Metrics at the declared per-class threshold', ylabel='Score', ylim=(0, 1))
for name, curve in CURVES.items():
    axes[1, 0].plot(curve['fpr'], curve['recall'], label=name)
    axes[1, 1].plot(curve['recall'], curve['precision'], label=name)
axes[1, 0].set(title='Pixel ROC per class', xlabel='FPR', ylabel='TPR')
axes[1, 1].set(title='Pixel precision-recall per class', xlabel='Recall', ylabel='Precision')
axes[1, 0].legend(fontsize=8); axes[1, 1].legend(fontsize=8)
save_figure(fig, OUTPUT_DIR / 'class_comparison.png')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
results[['nll_ms_mean', 'nll_ms_p95']].plot.bar(ax=axes[0])
axes[0].set(title='Observed backbone/NF/TTA latency', ylabel='ms/image')
errors = images.assign(error_fraction=(images.fp + images.fn) / (images.tp + images.fp + images.fn + images.tn))
for name, group in errors.groupby('class_name'):
    axes[1].hist(group.error_fraction, bins=25, alpha=.5, label=name)
axes[1].set(title='Fraction of wrong pixels per image', xlabel='(FP+FN)/pixels', ylabel='Images')
axes[1].legend(fontsize=8)
save_figure(fig, OUTPUT_DIR / 'latency_errors.png')

false_positives = images[(images.image_label == 0) & (images.image_prediction == 1)].sort_values('fp', ascending=False)
false_negatives = images[(images.image_label == 1) & (images.image_prediction == 0)].sort_values('fn', ascending=False)
display(Markdown('### Per-image false positives'))
display(false_positives[['class_name', 'path', 'image_score', 'fp', 'pixel_auroc']].head(20))
display(Markdown('### Per-image false negatives'))
display(false_negatives[['class_name', 'path', 'image_score', 'fn', 'recall', 'iou']].head(20))
false_positives.to_csv(OUTPUT_DIR / 'false_positives.csv', index=False)
false_negatives.to_csv(OUTPUT_DIR / 'false_negatives.csv', index=False)

In [ ]:
def json_ready(value):
    if isinstance(value, dict):
        return {key: json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


report = dict(created_at=datetime.now().isoformat(), python=sys.version, platform=platform.platform(),
              torch_version=torch.__version__, device=DEVICE, dataset=str(DATASET_ROOT),
              model_dir=str(MODEL_DIR), classes=CLASSES, scope='complete_test_descriptive_test_used_for_selection',
              pixel_backend=PIXEL_BACKEND, pixel_bins=N_BINS, bootstrap_images=BOOTSTRAP,
              warnings=['Histogram AUROC/AP are approximate in histogram mode.',
                        'Pixel CIs use reduced bins and are conditional on the fixed recipe/cohort.',
                        'Global AUPRO uses linear thresholds; zeros need auditing, not silent correction.',
                        'Thresholds and best F1 are test oracles.',
                        'Latency has no standardized warm-up and excludes I/O/post-processing.'],
              results=RESULTS)
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(json_ready(report), indent=2, allow_nan=False), encoding='utf-8')
(OUTPUT_DIR / 'summary.html').write_text(summary.to_html(float_format=lambda value: f'{value:.4f}'), encoding='utf-8')
try:
    with pd.ExcelWriter(OUTPUT_DIR / 'evaluation_tables.xlsx', engine='openpyxl') as writer:
        results.to_excel(writer, sheet_name='all_metrics')
        summary.to_excel(writer, sheet_name='summary_macro')
        images.to_excel(writer, sheet_name='per_image', index=False)
        subgroup_table.to_excel(writer, sheet_name='by_defect', index=False)
        micro.to_excel(writer, sheet_name='micro', index=False)
except ImportError:
    print('Optional Excel output unavailable: install openpyxl in the kernel. CSV/HTML/JSON were kept.')
from html import escape
parts = ['<html><meta charset="utf-8"><body><h1>Pixel-head evaluation</h1>',
         '<p>Complete test set, descriptive evaluation (the test set also selected the epochs). Scores are not probabilities.</p>',
         summary.to_html(float_format=lambda value: f'{value:.4f}'),
         '<img src="class_comparison.png" width="100%">']
for name in CLASSES:
    parts.extend([f'<h2>{escape(name)}</h2>',
                  f'<img src="{escape(name)}/curves_distributions_confusion.png" width="100%">',
                  f'<img src="{escape(name)}/heatmap_examples.png" width="100%">'])
parts.append('</body></html>')
(OUTPUT_DIR / 'report.html').write_text('\n'.join(parts), encoding='utf-8')
print('Report:', OUTPUT_DIR / 'report.html')
display(pd.DataFrame([dict(file=str(path.relative_to(OUTPUT_DIR)), size_kb=round(path.stat().st_size / 1024, 1))
                      for path in OUTPUT_DIR.rglob('*') if path.is_file()]))